# MeAJOR model type comparison
This notebook trains and compares three models on the MeAJOR dataset: TF-IDF + Logistic Regression, TF-IDF + Linear SVM, and a tabular Random Forest model using the full dataset headers (excluding sender, receiver, content_types, and source).

In [1]:
# Standard imports and project path setup
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

# Ensure repo root is on sys.path so imports from src/ work when running the notebook
repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

print('Repository root:', repo_root)

Repository root: C:\Users\brian\phishing-investigator


In [2]:
from src.ingestion.major_loader import load_major_training_dataset
from src.ingestion.major_loader_tabular import load_major_tabular_dataset

# Dataset path (update if you store it elsewhere)
DATASET_PATH = repo_root / 'data' / 'raw' / 'meajor_cleaned_preprocessed.csv'
print('Dataset path:', DATASET_PATH)

text_data = load_major_training_dataset(str(DATASET_PATH))
tabular_data = load_major_tabular_dataset(str(DATASET_PATH))

print('Text dataset shape:', text_data.shape)
print('Tabular dataset shape:', tabular_data.shape)

display(text_data.head())
display(tabular_data.head())
print('Label distribution:')
display(text_data['label'].value_counts())

Dataset path: C:\Users\brian\phishing-investigator\data\raw\meajor_cleaned_preprocessed.csv
Dropped 1 rows with missing labels
Dropped 1 rows with missing labels
Text dataset shape: (108684, 7)
Tabular dataset shape: (108684, 16)


,sender_domain,subject,body,text,label,source_dataset,source_type
0,enron.com,[organization] failover plan.,"hi [name], tonight we are rolling out a new re...",enron.com [organization] failover plan. hi [na...,0,meajor_cleaned_preprocessed.csv,meajor_csv
1,enron.com,re: intranet site,"[name] r these new? intranet site [name], we n...",enron.com re: intranet site [name] r these new...,0,meajor_cleaned_preprocessed.csv,meajor_csv
2,enron.com,fw: [organization] company information,"[name]/[name], we are currently trading under ...",enron.com fw: [organization] company informati...,0,meajor_cleaned_preprocessed.csv,meajor_csv
3,enron.com,new master physical,[name] and [name] - attached is a worksheet fo...,enron.com new master physical [name] and [name...,0,meajor_cleaned_preprocessed.csv,meajor_csv
4,enron.com,fw: [organization]/mirant gisb,fyi. below is a copy of my communication with ...,enron.com fw: [organization]/mirant gisb fyi. ...,0,meajor_cleaned_preprocessed.csv,meajor_csv


,sender_domain,receiver_domain,date,subject,body,urls,url_count,url_length_max,url_length_avg,url_subdom_max,url_subdom_avg,attachment_count,has_attachments,attachment_types,language,label
0,enron.com,enron.com,2001-06-29 09:37:04-05:00,[organization] failover plan.,"hi [name], tonight we are rolling out a new re...",,0.0,0.0,0.0,0.0,0.0,0.0,0,,en,0
1,enron.com,enron.com,2001-06-29 08:39:30-05:00,re: intranet site,"[name] r these new? intranet site [name], we n...",http://eastpower.dev.corp.enron.com/summary/pj...,3.0,60.0,58.0,3.0,3.0,0.0,0,,en,0
2,enron.com,enron.com;enron.com,2001-06-29 10:35:17-05:00,fw: [organization] company information,"[name]/[name], we are currently trading under ...",,0.0,0.0,0.0,0.0,0.0,0.0,0,,en,0
3,enron.com,enron.com;enron.com,2001-06-29 10:40:02-05:00,new master physical,[name] and [name] - attached is a worksheet fo...,,0.0,0.0,0.0,0.0,0.0,0.0,0,,en,0
4,enron.com,enron.com;enron.com;enron.com,2001-06-29 10:48:00-05:00,fw: [organization]/mirant gisb,fyi. below is a copy of my communication with ...,,0.0,0.0,0.0,0.0,0.0,0.0,0,,en,0


Label distribution:


label
0    60650
1    48034
Name: count, dtype: int64

In [3]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    print(f'=== {name} ===')
    print('Accuracy:', accuracy_score(y_test, predictions))
    print('Precision:', precision_score(y_test, predictions, average='weighted', zero_division=0))
    print('Recall:', recall_score(y_test, predictions, average='weighted', zero_division=0))
    print('F1:', f1_score(y_test, predictions, average='weighted', zero_division=0))
    print('Confusion matrix:')
    print(confusion_matrix(y_test, predictions))
    print('Classification report:')
    print(classification_report(y_test, predictions, zero_division=0))
    print()
    return predictions

# Text-only training data for the first two models
X_text = text_data['text']
y_text = text_data['label']
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y_text, test_size=0.2, random_state=42, stratify=y_text
)

logistic_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

svm_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
    ('classifier', LinearSVC(random_state=42, max_iter=20000)),
])

evaluate_model('TF-IDF + Logistic Regression', logistic_model, X_train_text, X_test_text, y_train_text, y_test_text)
evaluate_model('TF-IDF + Linear SVM', svm_model, X_train_text, X_test_text, y_train_text, y_test_text)

=== TF-IDF + Logistic Regression ===
Accuracy: 0.9798960298109215
Precision: 0.9799002194974795
Recall: 0.9798960298109215
F1: 0.9798976520756841
Confusion matrix:
[[11904   226]
 [  211  9396]]
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     12130
           1       0.98      0.98      0.98      9607

    accuracy                           0.98     21737
   macro avg       0.98      0.98      0.98     21737
weighted avg       0.98      0.98      0.98     21737


=== TF-IDF + Linear SVM ===
Accuracy: 0.9866126880434283
Precision: 0.9866256379472438
Recall: 0.9866126880434283
F1: 0.9866156093644732
Confusion matrix:
[[11964   166]
 [  125  9482]]
Classification report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     12130
           1       0.98      0.99      0.98      9607

    accuracy                           0.99     21737
   macro avg       0.99    

array([0, 1, 1, ..., 1, 1, 1], shape=(21737,))

In [5]:
# Tabular model using the full feature set except sender, receiver, content_types, and source
tabular_data['text'] = tabular_data['subject'].fillna('') + ' ' + tabular_data['body'].fillna('')

X_tab = tabular_data.drop(columns=['label'])
y_tab = tabular_data['label']

X_train_tab, X_test_tab, y_train_tab, y_test_tab = train_test_split(
    X_tab, y_tab, test_size=0.2, random_state=42, stratify=y_tab
)

tabular_preprocessor = ColumnTransformer([
    (
        'text',
        TfidfVectorizer(max_features=15000, ngram_range=(1, 2)),
        'text'
    ),
    (
        'urls',
        TfidfVectorizer(max_features=2000),
        'urls'
    ),
    (
        'cat',
        OneHotEncoder(handle_unknown='ignore'),
        ['sender_domain', 'receiver_domain', 'language', 'attachment_types']
    ),
    (
        'num',
        StandardScaler(),
        [
            'url_count',
            'url_length_max',
            'url_length_avg',
            'url_subdom_max',
            'url_subdom_avg',
            'attachment_count',
            'has_attachments'
        ]
    ),
], remainder='drop', sparse_threshold=0.3)

tabular_model = Pipeline([
    ('preprocessor', tabular_preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
])

evaluate_model('Tabular Random Forest', tabular_model, X_train_tab, X_test_tab, y_train_tab, y_test_tab)

=== Tabular Random Forest ===
Accuracy: 0.9881768413304504
Precision: 0.9881758851754301
Recall: 0.9881768413304504
F1: 0.9881760049982545
Confusion matrix:
[[12008   122]
 [  135  9472]]
Classification report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     12130
           1       0.99      0.99      0.99      9607

    accuracy                           0.99     21737
   macro avg       0.99      0.99      0.99     21737
weighted avg       0.99      0.99      0.99     21737




array([0, 1, 1, ..., 1, 1, 1], shape=(21737,))

**Notes:**
- The tabular model uses all allowed headers except sender, receiver, content_types, and source.
- The text models use the combined text representation from `sender_domain`, `subject`, and `body` via `load_major_training_dataset()`.
- Adjust the vectorizer and classifier hyperparameters to compare performance across different model types.